1. Load data

In [4]:
import sys
sys.path.append("../src")

import pandas as pd
import feature_engineering as fe

X_train, X_test, y_train, y_test = fe.load_train_test()

print("Train:", X_train.shape, " Test:", X_test.shape)
print("Fraud in train:", y_train.sum(), " Fraud in test:", y_test.sum())

Train: (200000, 14)  Test: (50000, 14)
Fraud in train: 384  Fraud in test: 96


2. Add new features

In [5]:
X_new = fe.add_features(X_train)
X_new[["amount_inr", "log_amount", "is_night", "is_high_amount", "same_bank", "same_age_group"]].head()

,amount_inr,log_amount,is_night,is_high_amount,same_bank,same_age_group
107369,1317,7.183871,0,0,0,0
127423,161,5.087596,0,0,0,1
181287,158,5.068904,0,0,0,0
215145,3692,8.214194,0,1,1,0
171409,1311,7.179308,0,0,1,0


New features: `log_amount`, `is_night` (hour 0-5), `is_high_amount` (above Rs 3500),
`same_bank`, `same_age_group`.
They use only the same row, so there is no data leakage.
There is no "transaction frequency" feature because the data has no user id.

3. Fraud rate for each new feature (train data)

In [6]:
X_new["fraud_flag"] = y_train

for col in ["is_night", "is_high_amount", "same_bank", "same_age_group"]:
    print(col)
    print((X_new.groupby(col)["fraud_flag"].mean() * 100).round(3))
    print()

is_night
is_night
0    0.191
1    0.204
Name: fraud_flag, dtype: float64

is_high_amount
is_high_amount
0    0.184
1    0.275
Name: fraud_flag, dtype: float64

same_bank
same_bank
0    0.196
1    0.168
Name: fraud_flag, dtype: float64

same_age_group
same_age_group
0    0.187
1    0.207
Name: fraud_flag, dtype: float64



Only `is_high_amount` shows a clear difference (0.275% fraud vs 0.184%).
The others are almost the same for 0 and 1. The new features carry very little signal.

4. Feature selection (keep the best 30)

In [7]:
import train_model as tm

data = tm.prepare_data()
print("Train shape after selection:", data["X_train"].shape)

Train shape after selection: (200000, 30)


`SelectKBest` keeps the 30 features most related to fraud. It is fit on train data only.